In [ ]:
from google.colab import files
uploaded = files.upload()


Saving best_deep_learning_model (2).keras to best_deep_learning_model (2).keras
Saving tokenizer (3).pkl to tokenizer (3).pkl
Saving sequence_config (2).json to sequence_config (2).json


In [ ]:
!pip install tensorflow -q


In [ ]:
import re
import json
import string
import pickle
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences


## 1. Loading the saved dump files


In [ ]:
best_dl_model = load_model("/content/best_deep_learning_model (2).keras")

with open("/content/tokenizer (3).pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("/content/sequence_config (2).json") as f:
    seq_config = json.load(f)
MAXLEN = seq_config["maxlen"]

print("Loaded deep learning model:", best_dl_model.name if hasattr(best_dl_model, 'name') else 'best_deep_learning_model')
print("Sequence maxlen:", MAXLEN)
print("Label mapping:", seq_config.get("label_mapping"))


Loaded deep learning model: childsafelens_bilstm
Sequence maxlen: 60
Label mapping: {'1': 'bullying (-1)', '0': 'non-bullying (0)'}


## 2. Preprocessing functions


In [ ]:
chat_words = {
    "u": "you", "ur": "your", "r": "are", "n": "and",
    "bcz": "because", "bcuz": "because", "coz": "because",
    "idk": "i do not know", "imo": "in my opinion", "imho": "in my humble opinion",
    "btw": "by the way", "lol": "laughing", "omg": "oh my god", "wtf": "what the fuck",
    "pls": "please", "plz": "please", "thx": "thanks", "fr": "for real",
    "bro": "brother", "msg": "message", "dm": "direct message",
    "cant": "can not", "dont": "do not", "wont": "will not", "aint": "is not",
    "ya": "you", "k": "okay", "ok": "okay",
    "kyu": "kyun", "kyun": "kyun", "tmhe": "tumhe", "tme": "tumhe", "mje": "mujhe",
    "kaha": "kahan", "krna": "karna", "krlo": "kar lo", "krdiya": "kar diya",
    "acha": "accha", "thik": "theek", "nhi": "nahi", "nai": "nahi",
    "hme": "humein", "apko": "aapko", "aapko": "aapko", "rha": "raha", "rhi": "rahi",
}

def clean_basic(text):
    text = str(text)
    text = re.sub(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", "", text
    )
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def expand_chat_words(text):
    words = text.split()
    return " ".join(chat_words.get(w, w) for w in words)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def tokenize_text(text):
    return text.split()

def preprocess_text(text, use_chat_words=True, use_punctuation_removal=True):
    text = clean_basic(text)
    if use_chat_words:
        text = expand_chat_words(text)
    if use_punctuation_removal:
        text = remove_punctuation(text)
    tokens = tokenize_text(text)
    return " ".join(tokens)

def clean_text_for_model(raw_text):
    """Produces the same 'clean_text' representation the tokenizer was fit on
    (chat-word expansion + punctuation strip, no lemmatization, no stopword removal)."""
    return preprocess_text(raw_text, use_chat_words=True, use_punctuation_removal=True)


## 3. Bi-LSTM model

In [ ]:
def classify_deep_learning(new_text, threshold=0.5):
    clean = clean_text_for_model(new_text)

    seq = tokenizer.texts_to_sequences([clean])
    padded = pad_sequences(seq, maxlen=MAXLEN, padding='post', truncating='post')

    prob = float(best_dl_model.predict(padded, verbose=0).ravel()[0])
    pred = "Cyberbullying" if prob >= threshold else "Non-Cyberbullying"

    print(f"Text: {new_text!r}")
    print(f"  Deep Learning (Bi-LSTM) -> {pred:<18} (probability: {prob:.4f})")
    print()

    return {
        "original_text": new_text,
        "cleaned_text": clean,
        "cyberbullying_probability": prob,
        "prediction": pred
    }


## 4. Testing


In [ ]:
classify_deep_learning('pay me 1 lakh or else i will leak all ur pics')


Text: 'pay me 1 lakh or else i will leak all ur pics'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9832)



{'original_text': 'pay me 1 lakh or else i will leak all ur pics',
 'cleaned_text': 'pay me lakh or else i will leak all your pics',
 'cyberbullying_probability': 0.9831823110580444,
 'prediction': 'Cyberbullying'}

In [ ]:
classify_deep_learning('hello')


Text: 'hello'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0121)



{'original_text': 'hello',
 'cleaned_text': 'hello',
 'cyberbullying_probability': 0.01214657537639141,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
classify_deep_learning('what are u doing')


Text: 'what are u doing'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0441)



{'original_text': 'what are u doing',
 'cleaned_text': 'what are you doing',
 'cyberbullying_probability': 0.04412468150258064,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
classify_deep_learning('lak tera akhri din hoga i will be killing u tom')


Text: 'lak tera akhri din hoga i will be killing u tom'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9710)



{'original_text': 'lak tera akhri din hoga i will be killing u tom',
 'cleaned_text': 'lak tera akhri din hoga i will be killing you tom',
 'cyberbullying_probability': 0.9709814786911011,
 'prediction': 'Cyberbullying'}

In [ ]:
classify_deep_learning('hey, kesi h kal mila h?')


Text: 'hey, kesi h kal mila h?'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.1602)



{'original_text': 'hey, kesi h kal mila h?',
 'cleaned_text': 'hey kesi h kal mila h',
 'cyberbullying_probability': 0.16021598875522614,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
classify_deep_learning('bkl apna muh khola na toh jinda nhi bachega')


Text: 'bkl apna muh khola na toh jinda nhi bachega'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9468)



{'original_text': 'bkl apna muh khola na toh jinda nhi bachega',
 'cleaned_text': 'bkl apna muh khola na toh jinda nahi bachega',
 'cyberbullying_probability': 0.9468383193016052,
 'prediction': 'Cyberbullying'}

In [ ]:
# Edit this list with whatever you want to test
test_sentences = [
    "tu bahut bakwas insaan hai",
    "thanks for helping me with the project",
    "i will find you and hurt you bastard",
    "accha weekend ho tumhara dost",
]

for s in test_sentences:
    classify_deep_learning(s)


Text: 'tu bahut bakwas insaan hai'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9429)

Text: 'thanks for helping me with the project'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0144)

Text: 'i will find you and hurt you bastard'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9204)

Text: 'accha weekend ho tumhara dost'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.1592)



In [ ]:
classify_deep_learning('shutt the fuck up')


Text: 'shutt the fuck up'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9766)



{'original_text': 'shutt the fuck up',
 'cleaned_text': 'shutt the fuck up',
 'cyberbullying_probability': 0.9765805602073669,
 'prediction': 'Cyberbullying'}

In [ ]:
classify_deep_learning('i m going to kill u tonight')


Text: 'i m going to kill u tonight'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0129)



{'original_text': 'i m going to kill u tonight',
 'cleaned_text': 'i m going to kill you tonight',
 'cyberbullying_probability': 0.012903973460197449,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
classify_deep_learning('mai aaj tujhe jaan se marne wali hu')


Text: 'mai aaj tujhe jaan se marne wali hu'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9562)



{'original_text': 'mai aaj tujhe jaan se marne wali hu',
 'cleaned_text': 'mai aaj tujhe jaan se marne wali hu',
 'cyberbullying_probability': 0.9561617374420166,
 'prediction': 'Cyberbullying'}

In [ ]:
classify_deep_learning('tere nudes h mere pass mujhe se mat uljana')


Text: 'tere nudes h mere pass mujhe se mat uljana'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9930)



{'original_text': 'tere nudes h mere pass mujhe se mat uljana',
 'cleaned_text': 'tere nudes h mere pass mujhe se mat uljana',
 'cyberbullying_probability': 0.9930150508880615,
 'prediction': 'Cyberbullying'}

In [ ]:
classify_deep_learning('I have your Nudes pics do not mess with me')


Text: 'I have your Nudes pics do not mess with me'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.8477)



{'original_text': 'I have your Nudes pics do not mess with me',
 'cleaned_text': 'i have your nudes pics do not mess with me',
 'cyberbullying_probability': 0.847698450088501,
 'prediction': 'Cyberbullying'}

In [ ]:
classify_deep_learning('hellooo how r u?')


Text: 'hellooo how r u?'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0212)



{'original_text': 'hellooo how r u?',
 'cleaned_text': 'helloo how are u',
 'cyberbullying_probability': 0.02123449184000492,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
classify_deep_learning('namaste kese ho tum')


Text: 'namaste kese ho tum'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0643)



{'original_text': 'namaste kese ho tum',
 'cleaned_text': 'namaste kese ho tum',
 'cyberbullying_probability': 0.06426633894443512,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
classify_deep_learning('mai toh bahr jaungi tum apna dekh lo')


Text: 'mai toh bahr jaungi tum apna dekh lo'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.2989)



{'original_text': 'mai toh bahr jaungi tum apna dekh lo',
 'cleaned_text': 'mai toh bahr jaungi tum apna dekh lo',
 'cyberbullying_probability': 0.29887303709983826,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
classify_deep_learning('mai kuch kha kar ati hu thode der me')


Text: 'mai kuch kha kar ati hu thode der me'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.2220)



{'original_text': 'mai kuch kha kar ati hu thode der me',
 'cleaned_text': 'mai kuch kha kar ati hu thode der me',
 'cyberbullying_probability': 0.2220006138086319,
 'prediction': 'Non-Cyberbullying'}

In [ ]:
# Edit this list with whatever you want to test
test_sentences = [
    "your posts are trash, nobody cares",
    "you explained the code really well, appreciate it",
    "tera DP bahut cringe hai",
    "happy birthday bro, enjoy your day",
    "stop talking, you sound so stupid",
    "great job on securing that internship",
]

for s in test_sentences:
    classify_deep_learning(s)


Text: 'your posts are trash, nobody cares'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9542)

Text: 'you explained the code really well, appreciate it'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0131)

Text: 'tera DP bahut cringe hai'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9377)

Text: 'happy birthday, enjoy your day'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.6722)

Text: 'stop talking, you sound so stupid'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.8414)

Text: 'great job on securing that internship'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.1538)



In [ ]:
# Edit this list with whatever you want to test
test_sentences = [
    "nobody wants you in this group",
    "congrats on your promotion, well deserved",
    "tumhari writing bahut weak hai, useless",
    "your presentation was clear and impressive",
    "i hate seeing your face online",
    "thank you for always supporting me",
]

for s in test_sentences:
    classify_deep_learning(s)


Text: 'nobody wants you in this group'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.8253)

Text: 'congrats on your promotion, well deserved'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0186)

Text: 'tumhari writing bahut weak hai, useless'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.5593)

Text: 'your presentation was clear and impressive'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0819)

Text: 'i hate seeing your face online'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.7394)

Text: 'thank you for always supporting me'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0233)



In [ ]:
# Edit this list with whatever you want to test
test_sentences = [
    "chal photos nikalte hai",
    "nice photo",
    "photo",
    "do u have sharukh khan's photos?",
    "maine aaj bahut ache pics li nature ki",
    "maine aaj bahut ache photos li nature ki",
    "you are amazing photographer tumahari liye har photos ache hote h",
    "meri ek photo kheecho is background me"
]

for s in test_sentences:
    classify_deep_learning(s)


Text: 'chal photos nikalte hai'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9221)

Text: 'nice photo'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0379)

Text: 'photo'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.0447)

Text: "do u have sharukh khan's photos?"
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.5617)

Text: 'maine aaj bahut ache pics li nature ki'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.5791)

Text: 'maine aaj bahut ache photos li nature ki'
  Deep Learning (Bi-LSTM) -> Non-Cyberbullying  (probability: 0.2613)

Text: 'you are amazing photographer tumahari liye har photos ache hote h'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9331)

Text: 'meri ek photo kheecho is background me'
  Deep Learning (Bi-LSTM) -> Cyberbullying      (probability: 0.9335)

